In [131]:
import torch
from torch.utils.data import Dataset
from torch_geometric.data import Data
from torch_geometric.nn.models import SchNet
from torch_geometric.loader import DataLoader
import torch.nn as nn
from ase.io import read as ase_read
from ase.data import atomic_numbers
from torch_geometric.datasets import QM9
from torch_geometric.nn.models import SchNet
from torch_geometric.nn.models import DimeNet
import os
from pathlib import Path
import pandas as pd
import yaml

In [132]:
XSD_PATH = "/Users/liuyihang/Desktop/dataset/C2H5OH.xsd"
ENERGY_EV = -382.123456


In [133]:
# config.py
import os
from dataclasses import dataclass, asdict
from datetime import datetime
import json
datetime.now().strftime("%Y-%m-%d---%H:%M")

'2025-11-30---09:52'

In [134]:
# ===== 读取 XSD =====
atoms = ase_read(XSD_PATH)              # ASE 会自动解析 Materials Studio 的 xsd
symbols = atoms.get_chemical_symbols()  # ['C','H',...]
pos = atoms.get_positions()

In [135]:
# ===== 构建 PyG 的 Data =====
z = torch.tensor([atomic_numbers[s] for s in symbols], dtype=torch.long)   # [N]
print(z)
pos = torch.tensor(pos, dtype=torch.float32)                                # [N,3]

data = Data(z=z, pos=pos)
if ENERGY_EV is not None:
    data.y = torch.tensor([ENERGY_EV], dtype=torch.float32)                 # [1]
Data

tensor([6, 6, 8, 1, 1, 1, 1, 1, 1])


torch_geometric.data.data.Data

In [136]:
loader = DataLoader([data], batch_size=1)

In [137]:
model = DimeNet(
    hidden_channels=128,
    out_channels= 1,
    num_blocks=2,
    num_bilinear=2,
    num_spherical=2,
    num_radial=2,
)


In [138]:
model.eval()
with torch.no_grad():
    for batch in loader:
        # PyG 的 SchNet 接口：forward(z, pos, batch) → 每个图一个标量（默认能量）
        pred_energy = model(batch.z, batch.pos, batch.batch)  # [batch_size, 1]
        print("Pred energy (arbitrary units before training):", pred_energy.squeeze().item())

Pred energy (arbitrary units before training): 0.0


In [139]:
TRAIN_DIR = "/Users/liuyihang/Desktop/dataset/train"
VAL_DIR = "/Users/liuyihang/Desktop/dataset/val"

TRAIN_XLSX = os.path.join(TRAIN_DIR, "train_energies.xlsx")
VAL_XLSX = os.path.join(VAL_DIR, "val_energies.xlsx")

BATCH_SIZE = 2
LR = 1e-3
EPOCHS = 50

# 如果你的能量是 Hartree 想转 eV，就把这个改成 27.2114；如果已经是 eV，保持 1.0
ENERGY_SCALE = 1.0

In [140]:
def load_energy_table(xlsx_path):
    """读取 Excel：第1列 = 名称，第2列 = 能量"""
    df = pd.read_excel(xlsx_path)
    name_col = df.columns[0]
    energy_col = df.columns[1]
    energy_dict = {}
    for _, row in df.iterrows():
        name = str(row[name_col]).strip()
        energy = float(row[energy_col]) * ENERGY_SCALE
        energy_dict[name] = energy
    return energy_dict
energy_dict = load_energy_table('/Users/liuyihang/Desktop/dataset/train/train_energies.xlsx')


def xsd_to_data(xsd_path, energy):
    """用 ASE 读取 xsd，转成 PyG 的 Data(z, pos, y)"""
    atoms = ase_read(xsd_path)
    symbols = atoms.get_chemical_symbols()
    positions = atoms.get_positions()  # Å

    z = torch.tensor([atomic_numbers[s] for s in symbols], dtype=torch.long)
    pos = torch.tensor(positions, dtype=torch.float32)
    y = torch.tensor([energy], dtype=torch.float32)

    return Data(z=z, pos=pos, y=y)

print(energy_dict)

{'C2H5OH': 7.212, 'C2H5O': 7.3342, 'C2H4': 7.3342, 'C2H5': 7.3342}


In [141]:
def build_graph_list(struct_dir, xlsx_path):
    """遍历目录下所有 xsd，按 Excel 匹配能量，生成 Data 列表"""
    energy_table = load_energy_table(xlsx_path)

    graph_list = []
    struct_dir = Path(struct_dir)
    for xsd_file in struct_dir.glob("*.xsd"):
        name = xsd_file.stem  # 去掉后缀
        if name not in energy_table:
            print(f"[WARN] {name} 在 {xlsx_path} 里找不到能量，跳过")
            continue
        energy = energy_table[name]
        print(energy, type(energy))
        data = xsd_to_data(str(xsd_file), energy)
        print(f"data.z is {data.z}")
        data.name = name
        graph_list.append(data)

    print(f"{struct_dir}: 读取到 {len(graph_list)} 个分子")
    return graph_list


In [142]:
class MoleculeDataset(Dataset):
    def __init__(self, data_list):
        self.data_list = data_list

    def __len__(self):
        return len(self.data_list)

    def __getitem__(self, idx):
        return self.data_list[idx]

In [143]:
def compute_mean_std(graphs):
    """计算能量的均值和方差，用于标准化（可选）"""
    ys = torch.stack([g.y for g in graphs], dim=0)  # [N,1]
    mean = ys.mean(dim=0)
    std = ys.std(dim=0)
    return mean, std


def standardize_graphs(graphs, mean, std):
    """把 Data.y 标准化为 (y - mean) / std"""
    std_safe = std.clone()
    std_safe[std_safe == 0] = 1.0
    for g in graphs:
        g.y = (g.y - mean) / std_safe
    return std_safe  # 用于反标准化时除回去


def evaluate(model, loader, device, y_mean, y_std):
    model.eval()
    mse_sum = 0.0
    mae_sum = 0.0
    n_samples = 0

    std_safe = y_std.clone()
    std_safe[std_safe == 0] = 1.0

    with torch.no_grad():
        for batch in loader:
            batch = batch.to(device)
            pred = model(batch.z, batch.pos, batch.batch)  # [B,1]
            pred = pred.squeeze(-1)
            # 反标准化回原始能量
            pred_real = pred * std_safe + y_mean
            y_real = batch.y * std_safe + y_mean

            mse = nn.functional.mse_loss(pred_real, y_real, reduction="sum")
            mae = nn.functional.l1_loss(pred_real, y_real, reduction="sum")

            mse_sum += mse.item()
            mae_sum += mae.item()
            n_samples += batch.num_graphs

    mse_mean = mse_sum / n_samples
    mae_mean = mae_sum / n_samples
    return mse_mean, mae_mean

In [144]:
def main():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print("Using device:", device)

    # 1) 构建 train / val 图数据
    train_graphs = build_graph_list(TRAIN_DIR, TRAIN_XLSX)
    val_graphs = build_graph_list(VAL_DIR, VAL_XLSX)
    
    print("train_graphs:", train_graphs)
    
    # 2) 计算 train 能量均值/方差，并标准化（提高训练稳定性）
    y_mean, y_std = compute_mean_std(train_graphs)
    print("Train y mean:", y_mean.item(), "std:", y_std.item())
    std_safe = standardize_graphs(train_graphs, y_mean, y_std)
    # val 也用同样的 mean/std 标准化
    standardize_graphs(val_graphs, y_mean, y_std)

    train_dataset = MoleculeDataset(train_graphs)
    val_dataset = MoleculeDataset(val_graphs)

    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

    # 3) 初始化 SchNet 模型
    model = SchNet(
        hidden_channels=128,
        num_filters=128,
        num_interactions=6,
        num_gaussians=50,
        cutoff=5.0,
        max_num_neighbors=64,
    ).to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=LR)
    loss_fn = nn.MSELoss()

    # 4) 训练循环
    for epoch in range(1, EPOCHS + 1):
        model.train()
        loss_sum = 0.0
        n_samples = 0

        for batch in train_loader:
            batch = batch.to(device)
            optimizer.zero_grad()

            pred = model(batch.z, batch.pos, batch.batch)  # [B,1]
            pred = pred.squeeze(-1)
            loss = loss_fn(pred, batch.y)                  # 此处用标准化后的 y
            loss.backward()
            optimizer.step()

            loss_sum += loss.item() * batch.num_graphs
            n_samples += batch.num_graphs

        train_loss = loss_sum / n_samples

        # 每个 epoch 验证一次（用反标准化后的 MSE/MAE）
        val_mse, val_mae = evaluate(model, val_loader, device, y_mean.to(device), std_safe.to(device))

        print(f"Epoch {epoch:03d} | train MSE(z) {train_loss:.6f} | "
              f"val MSE(real) {val_mse:.6f} | val MAE(real) {val_mae:.6f}")

    # 5) 保存模型和标准化参数
    save_obj = {
        "model_state": model.state_dict(),
        "y_mean": y_mean,
        "y_std": y_std,
        "energy_scale": ENERGY_SCALE,
    }
    torch.save(save_obj, "schnet_molecule_energy.pt")
    print("模型已保存到 schnet_molecule_energy.pt")

    
    for btach in train_loader:
        print(batch.y)
        pre = model(batch.z, batch.pos, batch.batch)
        print("The size of pre is: ", pre.size())
        print(pre)
        print("the pred.squeeze() is: ", pre.squeeze())
if __name__ == "__main__":
    main()

Using device: cpu
7.212 <class 'float'>
data.z is tensor([6, 6, 8, 1, 1, 1, 1, 1, 1])
7.3342 <class 'float'>
data.z is tensor([6, 6, 8, 1, 1, 1, 1, 1, 1])
7.3342 <class 'float'>
data.z is tensor([6, 6, 8, 1, 1, 1, 1, 1, 1])
7.3342 <class 'float'>
data.z is tensor([6, 6, 8, 1, 1, 1, 1, 1, 1])
/Users/liuyihang/Desktop/dataset/train: 读取到 4 个分子
7.262 <class 'float'>
data.z is tensor([6, 6, 8, 1, 1, 1, 1, 1, 1])
/Users/liuyihang/Desktop/dataset/val: 读取到 1 个分子
train_graphs: [Data(y=[1], pos=[9, 3], z=[9], name='C2H5OH'), Data(y=[1], pos=[9, 3], z=[9], name='C2H5O'), Data(y=[1], pos=[9, 3], z=[9], name='C2H4'), Data(y=[1], pos=[9, 3], z=[9], name='C2H5')]
Train y mean: 7.30364990234375 std: 0.061100006103515625
Epoch 001 | train MSE(z) 54.989990 | val MSE(real) 0.271373 | val MAE(real) 0.520935
Epoch 002 | train MSE(z) 43.105625 | val MSE(real) 0.010795 | val MAE(real) 0.103899
Epoch 003 | train MSE(z) 5.870150 | val MSE(real) 0.044452 | val MAE(real) 0.210836
Epoch 004 | train MSE(z) 19.7593